# Schrödinger 1D: cuaderno guiado (temporal + estacionario)

Este cuaderno tiene dos bloques:

1. evolución temporal de un paquete de onda,
2. problema estacionario de autovalores del Hamiltoniano.

Lectura física:

- $|\psi(x,t)|^2$ es densidad de probabilidad.
- La norma $\int |\psi|^2\,dx$ debe conservarse.
- En la evolución libre, el paquete puede ensancharse con el tiempo.


## A) Parte temporal

$$
i\psi_t = -\frac12\psi_{xx}.
$$

Con FFT, cada modo se actualiza multiplicando por un factor unitario.
Por eso el método conserva norma y evita crecimiento artificial de amplitud.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 20.0
N = 512
T = 1.4
dt = 0.004


In [ ]:
def build_problem(L=20.0, N=512, T=1.4, dt=0.004):
    nt = int(np.ceil(T / dt))
    dt = T / nt
    x = np.linspace(-L/2, L/2, N, endpoint=False)
    dx = x[1] - x[0]
    k = 2*np.pi*np.fft.fftfreq(N, d=dx)
    x0, sigma, k0 = -4.0, 0.85, 5.0
    psi = np.exp(-((x - x0)**2)/(2*sigma**2)) * np.exp(1j*k0*x)
    psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx)
    return x, dx, dt, nt, k, psi


def one_step_fft(psi, k, dt):
    phase = np.exp(-0.5j * (k**2) * dt)
    psi_hat = np.fft.fft(psi)
    psi_hat = phase * psi_hat
    return np.fft.ifft(psi_hat)


def solve_temporal(L=20.0, N=512, T=1.4, dt=0.004, store_every=6):
    x, dx, dt, nt, k, psi = build_problem(L=L, N=N, T=T, dt=dt)
    t_hist = [0.0]
    rho_hist = [np.abs(psi)**2]
    norm_hist = [float(np.sum(np.abs(psi)**2) * dx)]
    for n in range(nt):
        psi = one_step_fft(psi, k, dt)
        t_next = (n + 1) * dt
        if (n + 1) % store_every == 0 or (n + 1) == nt:
            t_hist.append(t_next)
            rho_hist.append(np.abs(psi)**2)
        norm_hist.append(float(np.sum(np.abs(psi)**2) * dx))
    return x, np.array(t_hist), np.array(rho_hist), np.array(norm_hist)


x, t_hist, rho_hist, norm_hist = solve_temporal(L=L, N=N, T=T, dt=dt, store_every=6)
print(f"norma inicial={norm_hist[0]:.6f}")
print(f"norma final={norm_hist[-1]:.6f}")


## B) Parte estacionaria

Construimos el Hamiltoniano discreto:

$$
H_h = -\frac12 D_{xx} + \operatorname{diag}(V).
$$

y resolvemos:

$$
H_h\phi_n = E_n\phi_n.
$$

Interpretación:

- $E_n$ son niveles de energía permitidos por el potencial.
- $\phi_n$ son modos espaciales asociados a cada nivel.
- En un pozo 1D con fronteras fijas, los modos presentan nodos crecientes al aumentar $n$.


Para el pozo infinito en $(0,1)$ usamos $V(x)=0$ en nodos interiores y condiciones $\phi(0)=\phi(1)=0$.

Con paso $h$, la matriz de segundo derivado se aproxima por:

$$
D_{xx}\approx
\frac{1}{h^2}
\begin{bmatrix}
-2 & 1 & 0 & \cdots & 0\\
1 & -2 & 1 & \ddots & \vdots\\
0 & \ddots & \ddots & \ddots & 0\\
\vdots & \ddots & 1 & -2 & 1\\
0 & \cdots & 0 & 1 & -2
\end{bmatrix}.
$$

Por tanto, $H_h$ es tridiagonal simétrica y se resuelve con `eigh`, que devuelve autovalores reales y autovectores ortogonales.


In [ ]:
def solve_stationary_well(n_int=180):
    h = 1.0 / (n_int + 1)
    x_int = np.linspace(h, 1.0 - h, n_int)
    main = (1.0 / h**2) * np.ones(n_int)
    off = (-0.5 / h**2) * np.ones(n_int - 1)
    H = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)
    evals, evecs = np.linalg.eigh(H)
    for j in range(6):
        norm_j = np.sqrt(np.sum(evecs[:, j] ** 2) * h)
        evecs[:, j] /= norm_j
    return x_int, evals, evecs

x_int, E, Phi = solve_stationary_well(n_int=180)
print("Primeros 6 niveles de energía:")
for j in range(6):
    print(f"n={j+1}: E={E[j]:.6f}")


In [ ]:
plt.figure(figsize=(8, 4))
for j in range(4):
    x_full = np.concatenate(([0.0], x_int, [1.0]))
    phi_full = np.concatenate(([0.0], Phi[:, j], [0.0]))
    plt.plot(x_full, phi_full, lw=2.0, label=f"phi_{j+1}(x)")
plt.title("Primeras autofunciones numéricas")
plt.xlabel("x")
plt.ylabel("phi_n(x)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()
